In [ ]:
import os
import numpy as np
import torch
from torchvision import transforms
from sklearn.cluster import KMeans
from huggingface_hub import login
from transformers import AutoModel, AutoImageProcessor
from PIL import Image
from tokencredentials import hf_token

hf_token = hf_token()
login(token=hf_token)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained("MahmoodLab/UNI", trust_remote_code=True)
model = model.to(device)
model.eval()

processor = AutoImageProcessor.from_pretrained("MahmoodLab/UNI", trust_remote_code=True)

def mock_macenko_normalize(patch_img):
    return patch_img

def extract_patient_embeddings(patch_paths):
    embeddings = []
    valid_paths = []
    
    for path in patch_paths:
        try:
            img = Image.open(path).convert("RGB")
            normalized_img = mock_macenko_normalize(img)
            
            inputs = processor(images=normalized_img, return_tensors="pt").to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
                if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
                    embedding = outputs.pooler_output.squeeze().cpu().numpy()
                else:
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
                    
            embeddings.append(embedding)
            valid_paths.append(path)
        except Exception as e:
            continue
            
    return np.array(embeddings), valid_paths

def perform_kmeans_reduction(embeddings, patch_paths, k=50):
    if len(embeddings) < k:
        return embeddings, patch_paths
        
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    
    representative_embeddings = []
    representative_paths = []
    
    for center in kmeans.cluster_centers_:
        distances = np.linalg.norm(embeddings - center, axis=1)
        closest_idx = np.argmin(distances)
        representative_embeddings.append(embeddings[closest_idx])
        representative_paths.append(patch_paths[closest_idx])
        
    return np.array(representative_embeddings), representative_paths

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
import os
import pandas as pd
import torch
import timm
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
from huggingface_hub import login
from tokencredentials import hf_token

login(token=hf_token())

csv_path = r"C:\Users\Aravind Kumar\Documents\Osteosarcoma Research\Implementation\master_dataset.csv"
df = pd.read_csv(csv_path)
image_paths = df["absolute_path"].tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    "hf-hub:MahmoodLab/uni", 
    pretrained=True, 
    init_values=1e-5, 
    dynamic_img_size=True
)
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

embeddings_dict = {}

for path in tqdm(image_paths):
    if pd.isna(path) or not os.path.exists(str(path)):
        continue
    
    try:
        image = Image.open(str(path)).convert("RGB")
        input_tensor = transform(image).unsqueeze(0).to(device)
        
        with torch.inference_mode():
            feature_emb = model(input_tensor)
            
        embeddings_dict[str(path)] = feature_emb.squeeze().cpu().numpy()
        
    except Exception as e:
        print(f"Error processing {path}: {e}")

np.save("uni_embeddings.npy", embeddings_dict)

c:\Users\Aravind Kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1144/1144 [1:17:35<00:00,  4.07s/it]


The next code block will group patches by their parent whole-slide image and calculate the nearest neighbors to build adjacency matrices. The resulting wsi_spatial_graphs.pt file will contain a dictionary of PyTorch tensors ready to be fed into the hypergraph network alongside the mRNA data.


In [3]:
import pandas as pd
import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors

csv_path = r"C:\Users\Aravind Kumar\Documents\Osteosarcoma Research\Implementation\master_dataset.csv"
df = pd.read_csv(csv_path)
embeddings_dict = np.load("uni_embeddings.npy", allow_pickle=True).item()

graphs = {}
grouped = df.groupby("ImageNumber")

for image_id, group in grouped:
    coords = group[["X.x", "X.y"]].values
    paths = group["absolute_path"].values
    
    valid_indices = [i for i, p in enumerate(paths) if str(p) in embeddings_dict]
    if not valid_indices:
        continue
        
    coords = coords[valid_indices]
    valid_paths = paths[valid_indices]
    
    node_features = np.stack([embeddings_dict[str(p)] for p in valid_paths])
    
    n_neighbors = min(8, len(coords))
    if n_neighbors < 2:
        continue
        
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, algorithm='ball_tree').fit(coords)
    _, indices = nbrs.kneighbors(coords)
    
    source_nodes = np.repeat(np.arange(len(coords)), n_neighbors)
    target_nodes = indices.flatten()
    
    edge_index = torch.tensor(np.vstack((source_nodes, target_nodes)), dtype=torch.long)
    x = torch.tensor(node_features, dtype=torch.float)
    pos = torch.tensor(coords, dtype=torch.float)
    

    label_str = group["classification"].iloc[0] 
    
    # 2. Map the string to a PyTorch-friendly integer
    label_map = {"Non-Tumor": 0, "Viable": 1, "Non-Viable-Tumor": 2, "viable: non-viable": 3}
    y_label = label_map.get(label_str, -1)
    
    graphs[image_id] = {
        "x": x,
        "edge_index": edge_index,
        "pos": pos,
        "y": torch.tensor([y_label], dtype=torch.long)
    }

torch.save(graphs, "wsi_spatial_graphs.pt")